## Identify Next Batch

In [0]:
%run "../00_common/01_environment_config"

In [0]:

# Databricks notebook source
# ============================================================
# F1 PROJECT - IDENTIFY PENDING BATCHES
# ============================================================

# Load environment configuration
# MAGIC %run ../00_common/01_environment_config

from pyspark.sql import functions as F
from delta.tables import DeltaTable

# Control table path
control_table_path = (
    f"{s3_root_path}/Control/Kalashri_increment/batch_control"
)

# Get landing batch folders
landing_batches = sorted([
    file.name.rstrip("/")
    for file in dbutils.fs.ls(landing_folder_path)
    if file.isDir()
    and file.name.rstrip("/") not in ("results", "sprints")
])

# Read batches already in progress or completed
if DeltaTable.isDeltaTable(spark, control_table_path):
    tracked_batches = [
        row.batch_id
        for row in (
            spark.read.format("delta")
            .load(control_table_path)
            .filter(
                F.col("status").isin("in_progress", "completed")
            )
            .select("batch_id")
            .distinct()
            .collect()
        )
    ]
else:
    tracked_batches = []

# Identify every pending batch
new_batches = sorted(
    set(landing_batches) - set(tracked_batches)
)

print(f"Landing batches: {landing_batches}")
print(f"Tracked batches: {tracked_batches}")
print(f"Pending batches: {new_batches}")
print(f"Total pending batches: {len(new_batches)}")

# Send all pending batches to the Databricks Job
dbutils.jobs.taskValues.set(
    key="batch_ids",
    value=new_batches
)

# ADD THIS
dbutils.jobs.taskValues.set(
    key="p_batch_id",
    value=new_batches[0] if new_batches else ""
)

dbutils.jobs.taskValues.set(
    key="has_batch",
    value="true" if new_batches else "false"
)

print("Batch selection completed.")


In [0]:

# # Databricks notebook source

# # ============================================================
# # F1 PROJECT - IDENTIFY NEXT BATCH
# # ============================================================
# # Purpose:
# # Identify the earliest unprocessed batch from the landing
# # folder and pass it to the next Databricks job task.
# # ============================================================

# # 1. Load existing environment configuration
# # MAGIC %run ../00-common/01.environment-config

# from pyspark.sql import functions as F
# from delta.tables import DeltaTable

# # ------------------------------------------------------------
# # 2. Define control table S3 path
# # ------------------------------------------------------------

# control_table_path = (
#     f"{s3_root_path}/Control/Kalashri_increment/batch_control"
# )

# # ------------------------------------------------------------
# # 3. Get batch folders from landing
# # ------------------------------------------------------------

# landing_batches = sorted([
#     file.name.rstrip("/")
#     for file in dbutils.fs.ls(landing_folder_path)
#     if file.isDir()
# ])

# # ------------------------------------------------------------
# # 4. Read tracked batches
# # ------------------------------------------------------------

# if DeltaTable.isDeltaTable(spark, control_table_path):

#     tracked_batches = [
#         row.batch_id
#         for row in (
#             spark.read
#                  .format("delta")
#                  .load(control_table_path)
#                  .filter(
#                      F.col("status").isin(
#                          "in_progress",
#                          "completed"
#                      )
#                  )
#                  .select("batch_id")
#                  .distinct()
#                  .collect()
#         )
#     ]

# else:
#     tracked_batches = []

# # ------------------------------------------------------------
# # 5. Identify earliest unprocessed batch
# # ------------------------------------------------------------

# new_batches = sorted(
#     set(landing_batches) - set(tracked_batches)
# )

# next_batch = new_batches[0] if new_batches else None

# # ------------------------------------------------------------
# # 6. Display batch-selection results
# # ------------------------------------------------------------

# print(f"Landing batches       : {landing_batches}")
# print(f"Tracked batches       : {tracked_batches}")
# print(f"Next batch to process : {next_batch}")

# # ------------------------------------------------------------
# # 7. Pass values to the next Databricks job task
# # ------------------------------------------------------------

# if next_batch is None:

#     dbutils.jobs.taskValues.set(
#         key="p_batch_id",
#         value=""
#     )

#     dbutils.jobs.taskValues.set(
#         key="has_batch",
#         value="false"
#     )

# else:

#     dbutils.jobs.taskValues.set(
#         key="p_batch_id",
#         value=next_batch
#     )

#     dbutils.jobs.taskValues.set(
#         key="has_batch",
#         value="true"
#     )
